In [ ]:

using DelimitedFiles
#using LightGraphs
using LinearAlgebra 
using Plots
using Random
#using BenchmarkTools
using Distributions
using StatsBase
using  OrdinaryDiffEq ###DifferentialEquations
using DiffEqCallbacks
using WebIO
using Peaks
using Plots

##using DifferentialEquations

In [ ]:
N=1000;

# setting up time steps and integration intervals

dt = 0.01 # time step
dts = 0.1 # save time
ti = 0.0
tf =300
tt =0
nt = Int(div(tt,dts))
nf = Int(div(tf,dts))
vector_t=tt:dts:tf
tspan = (ti, tf);

# For Positive and Negative J value with fixed F=1.0

In [ ]:
J=1.0
K=2.5
F=1.0
Δ=collect(0.5:0.1:0.5)
p = [Δ, F, J, K]


In [ ]:


function swarmforced!(du, u, p, t)

       u1 = @view u[1:N] ##x 
       du1 = @view du[1:N] ## dx
       u2 = @view u[N+1:2*N] ## θ
       du2 = @view du[N+1:2*N] ## dθ
       

        Δ,F,J,K=p

        z1 = Array{Complex{Float64},1}(undef, 1)
        z1= mean(exp.((u1+u2)*1im))
         
        z2 = Array{Complex{Float64},1}(undef, 1)
        z2= mean(exp.((u1-u2)*1im))
        

        ####### equ of motion\n",
        @. du1  = J/2 * (( imag(z1*exp((-1im)*(u1+u2))))+( imag(z2*exp((-1im)*(u1-u2))))) 

        @. du2 = Δ - F*sin(u2) + K/2 * (( imag(z1*exp((-1im)*(u1+u2))))-( imag(z2*exp((-1im)*(u1-u2))))) 
    
        return du 
    
       end;

In [ ]:
u0=[rand(N)*2*pi .- pi; rand(N)*2*pi .- pi];

#u0=[(rand(N)*2*pi )*1; (rand(N)*2*pi )*1];



prob = ODEProblem(swarmforced!, u0, tspan,p);

    ##############################
saved_values1= SavedValues(Float64,ComplexF64)
       function saver1(u,t,integrator)
            _pp=u[1:N]
            _qq=u[N+1:2*N]
            out1= mean(exp.((_pp + _qq)*1im))
       end
     cb1 = SavingCallback(saver1, saved_values1,saveat=tt:dts:tf)
################################
################################
      saved_values2= SavedValues(Float64,ComplexF64)
       function saver2(u,t,integrator)
           _pp=u[1:N]
           _qq=u[N+1:2*N]
          out2=  mean(exp.((_pp - _qq)*1im))
       end
     cb2 = SavingCallback(saver2, saved_values2,saveat=tt:dts:tf)
################################
################################
     saved_values3= SavedValues(Float64,ComplexF64)
     function saver3(u,t,integrator)
           _pp=u[1:N]
        out3= mean(exp.((_pp)*1im))
     end
   cb3 = SavingCallback(saver3, saved_values3,saveat=tt:dts:tf)
####################################
####################################    
    saved_values4= SavedValues(Float64,ComplexF64)
     function saver4(u,t,integrator)
           _qq= u[N+1:2*N]
        out4= mean(exp.((_qq)*1im))
     end
   cb4 = SavingCallback(saver4, saved_values4,saveat=tt:dts:tf)
####################################

####################################    
    saved_values5= SavedValues(Float64,Float64)
     function saver5(u,t,integrator)
           duc= swarmforced!(zeros(size(u)),u,integrator.p,t)
            _pp=duc[1:N]
           _qq=duc[N+1:2*N]
        out5= mean(sqrt.(((_pp).^2)+((_qq).^2)))
     end
   cb5 = SavingCallback(saver5, saved_values5,saveat=tt:dts:tf)
####################################   

####################################    
    saved_values6= SavedValues(Float64,Float64)
     function saver6(u,t,integrator)
           duc= swarmforced!(zeros(size(u)),u,integrator.p,t)
            _pp=duc[1:N]   
        out6= mean(sqrt.((_pp).^2))
     end
   cb6 = SavingCallback(saver6, saved_values6,saveat=tt:dts:tf)
####################################   

####################################    
    saved_values7= SavedValues(Float64,Float64)
     function saver7(u,t,integrator)
           duc= swarmforced!(zeros(size(u)),u,integrator.p,t)
           _qq=duc[N+1:2*N]
        out7= mean(sqrt.((_qq).^2))
     end
   cb7 = SavingCallback(saver7, saved_values7,saveat=tt:dts:tf)
####################################  

    saved_values8=  SavedValues(Float64,ComplexF64)
     function saver8(u,t,integrator)
               _pp = u[1:N]
              out8 = mean(exp.(2im .*_pp))
     end
   cb8 = SavingCallback(saver8, saved_values8,saveat=tt:dts:tf)
 ###################################  

    saved_values9= SavedValues(Float64,ComplexF64)
     function saver9(u,t,integrator)
               _qq=u[N+1:2*N]
              out9 = mean(exp.(2im .*_qq))
        end
   cb9 = SavingCallback(saver9, saved_values9,saveat=tt:dts:tf)
 ###################################  


 cbs = CallbackSet(cb1, cb2, cb3, cb4, cb5, cb6, cb7, cb8, cb9 );
    


In [ ]:
@time sol = solve(prob, RK4(), dt= 0.01,callback=cbs,saveat=tt:dts:tf);

In [ ]:
x = sol[1:N,:];
θ = (sol[N+1:2*N,:] );

In [ ]:
# Define the range of wavelengths
# Create a scatter plot


   theta = range(0, stop=2π, length=200)
    x1 = cos.(theta)
    y1 = sin.(theta)
    # Plot the unit circle
    p1=plot(x1, y1,color="black",linewidth=0.5)
    
    colors=θ[:,end]
scatter!(cos.(x[:, end]), sin.(x[:, end]), color=colors, markersize=8, marker_z=colors, c=:jet, legend=false, label="",colorbar=true, clim=(-π,π))
    xlims!(-1.1, 1.1)
    ylims!(-1.1, 1.1)



p2= scatter(x[:, end],θ[:, end],color="blue",markersize=8,label="")

#scatter!([x[34,end-100:end]],[θ[34,end-100:end]],color="red",markersize=1,label="")
#scatter!(x[34,end-100:end],[θ[34,end-100:end]],color="red",markersize=10,label="")


xlims!(-4, 4)  # Set the x-axis limits
ylims!(-4, 4)  # Set the y-axis limits
title!("N=$N, J=$J, K=$K, Δ=$Δ, F=$F")

#scatter!(x[:, end],mod2pi.(u0[N+1:2*N, end]).-π,color="pink",markersize=4,label="")
# Scatter a point at the mean of x[:, end]
#scatter!([mean(x[:, end])], [0], color="red", markersize=10, label="")

# Setting axes to the origin
#plot!()  # Create a plot object
hline!([0], lw=0.8, lc=:black,label="")  # Horizontal line at y = 0
vline!([0], lw=0.8, lc=:black,label="")  # Vertical line at x = 0

# Customize the plot appearance
xlabel!("x")
ylabel!("θ")
plot!(grid=false)
#plot!(box=false)
#plot!(framestyle=:none)
#legend(loc=:topright)

p3= scatter(mod2pi.(sol[1:N, end]+sol[N+1:2*N, end]).-π, mod2pi.(sol[1:N, end]-sol[N+1:2*N, end]).-π,color="blue",markersize=2,label="")
xlims!(-4, 4)  # Set the x-axis limits
ylims!(-4, 4)  # Set the y-axis limits
#title!("N=$N, J=$J, K=$K, Δ=$Δ, F=$F")

#scatter!(x[:, end],mod2pi.(u0[N+1:2*N, end]).-π,color="pink",markersize=4,label="")
# Scatter a point at the mean of x[:, end]
#scatter!([mean(x[:, end])], [0], color="red", markersize=10, label="")

# Setting axes to the origin
#plot!()  # Create a plot object
hline!([0], lw=0.8, lc=:black,label="")  # Horizontal line at y = 0
vline!([0], lw=0.8, lc=:black,label="")  # Vertical line at x = 0

# Customize the plot appearance
xlabel!("ξ")
ylabel!("η")
plot!(grid=false)
#plot!(box=false)
#plot!(framestyle=:none)
#legend(loc=:topright)

ppp=plot(p1, p2, layout=(1, 2),size=(1200, 400),grid=false)  


#savefig("N=$N, J=$J, K=$K, Δ=$Δ, F=$F.png")

display(ppp)


In [ ]:

using Interact
#using Blink
index=1 #Int(floor(0.5*size(x,2)))
@manipulate for t in index:1:size(x,2)
    
    theta = range(0, stop=2π, length=200)
    x1 = cos.(theta)
    y1 = sin.(theta)
    # Plot the unit circle
    p1=plot(x1, y1,color="black",linewidth=0.5)
    
    colors=θ[:,t]
    scatter!(cos.(x[:, t]), sin.(x[:, t]), color=colors, markersize=6, marker_z=colors, c=:jet, legend=false, label="",colorbar=true,clim=(-π,π))
    xlims!(-1.1, 1.1)
    ylims!(-1.1, 1.1)
    title!("N=$N, J=$J, K=$K, Δ=$Δ, F=$F")
    
    p2= scatter(x[:,t],θ[:,t],color="blue",markersize=6,label="")
    title!("t = $t")
    xlims!(-π, π)
    ylims!(-π, π)
    xlabel!("x")
    ylabel!("θ")
#    scatter!([mean(x[:, end])], [0], color="red", markersize=10, label="")
    # Setting axes to the origin
#plot!()  # Create a plot object
hline!([0], lw=0.8, lc=:black,label="")  # Horizontal line at y = 0
vline!([0], lw=0.8, lc=:black,label="")  # Vertical line at x = 0

    
    
    
    plot!(grid=false)
plot!(box=false)
    
plot(p1, p2, layout=(1, 2),size=(900, 400),grid=false)    
#plot(p2,size=(900, 500),grid=false)
    #plot!(framestyle=:none)
end
#savefig("N=$N, J=$J, K=$K, Δ=$Δ, F=$F.png")


In [ ]:
S_plus=saved_values1.saveval;
S_minus=saved_values2.saveval;
Y=saved_values3.saveval;
Z=saved_values4.saveval;
vel=saved_values5.saveval;
vel_x=saved_values6.saveval;
vel_y=saved_values7.saveval;
D_x=saved_values8.saveval;
D_Θ=saved_values9.saveval;


In [ ]:
abs_Sp=abs.(S_plus)
abs_Sm=abs.(S_minus)
abs_Q=abs.(Y);
abs_R=abs.(Z);
abs_D_x=abs.(D_x);
abs_D_Θ=abs.(D_Θ);


In [ ]:
s_plus1= max.(abs_Sp, abs_Sm);
s_minus1=min.(abs_Sp, abs_Sm);

In [ ]:
a=ones(size( vector_t))

In [ ]:
using Plots
gr()  # or your preferred backend

# Sample code (you must already have vector_t, abs_Sp, abs_Sm, N, J, K, Δ, F defined)

# Plot S⁺ and S⁻
p = plot(vector_t, max.(abs_Sp, abs_Sm), color="red", linewidth=3, label="S⁺")
plot!(p, vector_t, min.(abs_Sp, abs_Sm), color="blue", linewidth=3, label="S⁻")
plot!(p, vector_t,a, color="pink", linewidth=3, label="y=1")

# Decorations
title!(p, "N=$N, J=$J, K=$K, Δ=$Δ, F=$F")
ylims!(p, 0, 1.0)
xlabel!(p, "t")
plot!(p, grid=false)

# Display
display(p)

# Save the figure
#savefig(p, "Splus_Sminus,N=$N,J=$J,K=$K,Δ=$Δ,F=$F.png")


In [ ]:
using Plots
gr()  # or your preferred backend

# Sample code (you must already have vector_t, abs_Sp, abs_Sm, N, J, K, Δ, F defined)

# Plot S⁺ and S⁻
p = plot(vector_t, abs_D_x, color="red", linewidth=3, label="D_x")
plot!(p, vector_t, abs_D_Θ, color="blue", linewidth=3, label="D_Θ")
plot!(p, vector_t,a, color="black", linewidth=3, label="y=1")

# Decorations
title!(p, "N=$N, J=$J, K=$K, Δ=$Δ, F=$F")
ylims!(p, 0, 1.0)
xlabel!(p, "t")
plot!(p, grid=false)

# Display
display(p)

# Save the figure
#savefig(p, "abs_D_x,D_Θ,N=$N,J=$J,K=$K,Δ=$Δ,F=$F.png")


In [ ]:
using Plots
gr()  # Set your preferred backend

# Plot Q
p = plot(vector_t, abs_Q, color="green", linewidth=3, label="Q")

# Add R to the same plot
plot!(p, vector_t, abs_R, color="black", linewidth=3, label="R")
plot!(p, vector_t,a, color="pink", linewidth=3, label="y=1")

# Customize plot
title!(p, "N=$N, J=$J, K=$K, Δ=$Δ, F=$F")
ylims!(p, 0, 1.0)
xlabel!(p, "t")
plot!(p, grid=false)

# Display the plot
display(p)

# Save the figure as PNG
#savefig(p, "Q_R,N=$N,J=$J,K=$K,Δ=$Δ,F=$F.png")


In [ ]:
p4=plot(vector_t, vel, color="magenta", linewidth=3, label="vel")
title!("N=$N, J=$J, K=$K, Δ=$Δ, F=$F")
plot!(grid=false)
p5=plot!(vector_t, vel_x, color="blue", linewidth=3, label="vel_x")
p6=plot!(vector_t, vel_y, color="red", linewidth=3, label="vel_θ")
#ylims!(0, 0.2)